# Assignment # 04

## TechGadgets Retailer

### Task 1.1: Data Overview and Cleaning

The dataset was cleaned using a checklist-based approach to ensure data consistency and readiness for analysis.

The following steps were applied:

- Datetime conversion: `Purchase Date` and `Delivery Date` columns were converted to datetime format to enable accurate time-based calculations.
- Duplicate removal: Duplicate rows were removed to avoid double counting and redundancy in the analysis.
- Logical consistency check: Rows where `Delivery Date` occurred before `Purchase Date` were removed to maintain temporal logic.
- Handling missing values: Missing values were checked across all columns. Rows with missing values in the `Profit` column were removed, as `Profit` is critical for profitability-related analysis.
- Text data standardization: Text-based columns (`Customer Segment`, `Gadget Type`, `City`, `State`, `Region`) were stripped of leading/trailing whitespace and standardized in casing.
- Data type optimization: Suitable text-based columns (`Customer Segment`, `Gadget Type`, `City`, `State`, `Region`) were converted to the `category` data type to improve memory efficiency and clarity.
- Outlier removal: Extreme values in the `Profit` column were removed by keeping only rows where `Profit` lies between **-6000** and **10000**, avoiding distortions in statistical analyses.
- Index reset: After cleaning, the dataset index was reset to maintain consistent row ordering.
- Retention of Gadget ID nulls: Rows with missing `Gadget ID` values were retained, as this field is not required for the analysis tasks in Assignment 4 and excluding them would unnecessarily reduce the dataset size.

In [1]:
#Task1.1

import pandas as pd

# Loading dataset
df = pd.read_csv("techgadgets_data.csv")

# Looking dataset info
df.info()

# Checking for duplicated rows
print("\nDuplicated rows:", df.duplicated().sum())

# Converting date columns to datetime
df["Purchase Date"] = pd.to_datetime(df["Purchase Date"], dayfirst=True, errors="coerce")
df["Delivery Date"] = pd.to_datetime(df["Delivery Date"], dayfirst=True, errors="coerce")

# Removing duplicate rows
df = df.drop_duplicates()

# Removing rows where Delivery Date is earlier than Purchase Date
df = df[df["Delivery Date"] >= df["Purchase Date"]]

# Removing rows with missing Profit (critical for analysis)
df = df.dropna(subset=["Profit"])

# Standarding text columns (strip whitespace and title case)
text_columns = ["Customer Segment", "Gadget Type", "City", "State", "Region"]
for col in text_columns:
    df[col] = df[col].astype(str).str.strip().str.title()

# Converting selected columns to categorical dtype
categorical_cols = ["Customer Segment", "Gadget Type", "City", "State", "Region"]
for col in categorical_cols:
    df[col] = df[col].astype("category")

# Basic statistics before cleaning
mean_profit = df["Profit"].mean()
median_profit = df["Profit"].median()
min_profit = df["Profit"].min()
max_profit = df["Profit"].max()

print("Profit statistics after filtering:")
print(f"Mean:   {mean_profit:.4f}")
print(f"Median: {median_profit:.4f}")
print(f"Min:    {min_profit:.4f}")
print(f"Max:    {max_profit:.4f}")

# Removing extreme outliers in Profit (< -6000 or > 10000)
df = df[(df["Profit"] >= -6000) & (df["Profit"] <= 10000)]

# Checking for missing values
print("\nMissing values per column:")
print(df.isnull().sum())

# Resetting index
df = df.reset_index(drop=True)

# Basic statistics after cleaning
mean_profit = df["Profit"].mean()
median_profit = df["Profit"].median()
min_profit = df["Profit"].min()
max_profit = df["Profit"].max()

print("\nProfit statistics after filtering:")
print(f"Mean:   {mean_profit:.4f}")
print(f"Median: {median_profit:.4f}")
print(f"Min:    {min_profit:.4f}")
print(f"Max:    {max_profit:.4f}")

# Copying cleaned dataframe for analysis
df_cleaned = df.copy()

# Looking cleaned dataset info
df_cleaned.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Purchase Date     9994 non-null   object 
 1   Delivery Date     9994 non-null   object 
 2   Customer Segment  9994 non-null   object 
 3   City              9994 non-null   object 
 4   State             9795 non-null   object 
 5   Region            9994 non-null   object 
 6   Gadget ID         9601 non-null   object 
 7   Gadget Type       9994 non-null   object 
 8   Profit            9799 non-null   float64
dtypes: float64(1), object(8)
memory usage: 702.8+ KB

Duplicated rows: 1
Profit statistics after filtering:
Mean:   390.4571
Median: 7.8225
Min:    -6599.9780
Max:    505583.6546

Missing values per column:
Purchase Date         0
Delivery Date         0
Customer Segment      0
City                  0
State                 0
Region                0
Gadget ID           385
G

## Task 1.2: Measurement Scales of Attributes

Each attribute in the dataset was evaluated based on its nature and usage.  
Below is a classification of the measurement scales:

- **Nominal**: Customer Segment, City, State, Region, Gadget ID, Gadget Type  
- **Interval**: Purchase Date, Delivery Date  
- **Ratio**: Profit  

Attributes such as `Profit` are ratio-scaled because they have a true zero and allow meaningful numerical operations, including ratios.  
Categorical variables such as `Customer Segment`, `City`, and `Gadget Type` are nominal because they represent distinct categories with no inherent order.  
`Purchase Date` and `Delivery Date` are interval-scaled as the time between dates is meaningful, but the scale lacks a true zero point.


### Task 2.1: Delivery Time vs Profit
#### Hypotheses
- **Null Hypothesis (H₀):** There is no significant difference in profit between short and long delivery groups.
- **Alternative Hypothesis (H₁):** There is a significant difference in profit between short and long delivery groups.
#### Grouping Logic
- The median delivery time is **6 days**.
- Two groups were created:
  - **Short Delivery:** Delivery Time ≤ 6 days
  - **Long Delivery:** Delivery Time > 6 days
#### Levene’s Test for Homogeneity of Variance
- p-value = 0.511 → p > 0.05  
- Assumption of equal variances is met.
#### Independent T-Test Results
- t-statistic = -0.723  
- p-value = 0.470 → p > 0.05
#### Conclusion
Since the p-value is greater than 0.05, we **fail to reject the null hypothesis**.  
There is **no statistically significant difference** in profit between short and long delivery groups.
This suggests that faster or slower delivery does not significantly impact profit for TechGadgets.


In [2]:
import numpy as np
from scipy.stats import ttest_ind, levene

# Defining the median delivery time as the threshold
threshold = df_cleaned['Delivery Date'].median()

# Creating a new column to classify deliveries as Short or Long
df_cleaned['Delivery Group'] = np.where(df['Delivery Date'] <= threshold, 'Short', 'Long')

# Separating profit values based on delivery group
profit_short = df_cleaned[df_cleaned['Delivery Group'] == 'Short']['Profit']
profit_long = df_cleaned[df_cleaned['Delivery Group'] == 'Long']['Profit']

# Levene’s Test to check homogeneity of variance
levene_stat, levene_p = levene(profit_short, profit_long)

# Performing independent t-test
# If variances are equal (p >= 0.05), assume equal variance; otherwise, use Welch's t-test
equal_var = levene_p >= 0.05
t_stat, t_p = ttest_ind(profit_short, profit_long, equal_var=equal_var)

# Printing results
print("Levene p-value:", levene_p)
print("Equal variance assumed?", equal_var)
print("T-test statistic:", t_stat)
print("T-test p-value:", t_p)


Levene p-value: 0.5110534814100551
Equal variance assumed? True
T-test statistic: -0.7225205043371371
T-test p-value: 0.4699918583432303


### Task 2.2: Profit Differences Between Smartphone And Other Gadget Types

#### Hypotheses
- **Null Hypothesis (H₀):** There is no significant difference in profit among the three gadget types.
- **Alternative Hypothesis (H₁):** There is a significant difference in profit among at least two gadget types.

#### Data Preparation
- Rows with missing values in `Profit` or `Gadget Type` were removed.
- `Profit` values were grouped by `Gadget Type`: **Tablet**, **Laptop**, and **Smartphone**.

#### Levene’s Test for Homogeneity of Variance
- Levene’s Test was performed to assess whether the variance in profits is equal across the three groups.
- **Result**: p-value = **5.41 × 10⁻³⁰** → p < 0.05  
- The assumption of equal variances is **violated**.

#### Welch’s ANOVA Results
- Welch’s ANOVA was applied since the assumption of equal variances was not met.
- **Test statistic**: 6.77  
- **p-value**: **1.69 × 10⁻¹¹** → p < 0.05

#### Conclusion
Since the p-value is significantly less than 0.05, we **reject the null hypothesis**.  
There is a **statistically significant difference in profit** among at least two gadget types.  
This indicates that the type of gadget (Tablet, Laptop, or Smartphone) has a meaningful impact on the profit generated.


In [3]:
import numpy as np
from scipy.stats import ttest_ind, levene

# Creating a new column to classify gadget type as Smartphone or Other
df_cleaned['Gadget Group'] = np.where(df_cleaned['Gadget Type'] == 'Smartphone', 'Smartphone', 'Other')

# Separating profit values based on gadget group
profit_smartphone = df_cleaned[df_cleaned['Gadget Group'] == 'Smartphone']['Profit']
profit_other = df_cleaned[df_cleaned['Gadget Group'] == 'Other']['Profit']

# Levene’s Test to check homogeneity of variance
levene_stat, levene_p = levene(profit_smartphone, profit_other)

# Performing independent t-test or welch test
equal_var = levene_p >= 0.05
t_stat, t_p = ttest_ind(profit_smartphone, profit_other, equal_var=equal_var)

# Printing results
print("Levene p-value:", levene_p)
print("Equal variance assumed?", equal_var)
print("T-test statistic:", t_stat)
print("T-test p-value:", t_p)


Levene p-value: 5.413246225620069e-30
Equal variance assumed? False
T-test statistic: 6.77121214765158
T-test p-value: 1.6884161175402856e-11


### Task 2.3: Profit vs All Gadget Types

#### **Hypotheses**
- **Null Hypothesis (H₀):** The average profit is the same for all gadget types (Laptop, Tablet, Smartphone).
- **Alternative Hypothesis (H₁):** At least one gadget type generates a significantly different average profit.
#### **Preconditions**
- Dependent variable (`Profit`) is continuous
- Independent variable (`Gadget Type`) has **three categories**
- Observations are independent
- Sample size > 3000 → Normality assumed via Central Limit Theorem 
- Levene’s Test used to assess variance homogeneity
#### **Levene’s Test for Homogeneity of Variance**
- **p-value = 1.84e-33** → p < 0.05  
  **Equal variances not assumed**  
  Proceed with **Welch's ANOVA**
#### **Welch’s ANOVA Results**
- **F-statistic = 28.99**  
- **p-value = 3.23e-13**  
- **Degrees of freedom = (2.0, 3551.91)**  
- **Group means**:  
  - Tablet: 19.41  
  - Laptop: 81.35  
  - Smartphone: 7.78
#### **Conclusion**
Since the p-value is **much smaller than 0.05**, we **reject the null hypothesis**.  
There is a **statistically significant difference** in the average profit among at least two gadget types.

This result suggests that **profitability is not equal across Laptops, Tablets, and Smartphones**.

In [4]:
from scipy.stats import levene
from statsmodels.stats.oneway import anova_oneway

# Preparing the data (remove NaN)
df_anova = df_cleaned.dropna(subset=["Profit", "Gadget Type"])

# Levene Test (for variance homogeneity)
group1 = df_anova[df_anova["Gadget Type"] == "Tablet"]["Profit"]
group2 = df_anova[df_anova["Gadget Type"] == "Laptop"]["Profit"]
group3 = df_anova[df_anova["Gadget Type"] == "Smartphone"]["Profit"]

levene_stat, levene_p = levene(group1, group2, group3, center="median")
print("Levene Test p-value:", levene_p)

# Welch ANOVA using direct Series inputs
anova_result = anova_oneway(
    df_anova["Profit"],
    df_anova["Gadget Type"],
    use_var="unequal"
)

print(anova_result)

Levene Test p-value: 1.8444374254006455e-33
statistic = 28.994479414269787
pvalue = 3.2325323452954334e-13
df = (2.0, np.float64(3551.908220333658))
df_num = 2.0
df_denom = 3551.908220333658
nobs_t = 9749.0
n_groups = 3
means = [19.41486332 81.34677344  7.77850907]
nobs = [5881. 1807. 2061.]
vars_ = [ 27147.81785748 160675.52413038  17138.72729639]
use_var = unequal
welch_correction = True
tuple = (np.float64(28.994479414269787), np.float64(3.2325323452954334e-13))


### Task 2.4: Smartphone Purchases vs Customer Segment

#### **Hypotheses**
- **Null Hypothesis (H₀):** There is no association between customer segment and smartphone purchases.
- **Alternative Hypothesis (H₁):** There is a significant association between customer segment and smartphone purchases.
#### **Test Used**
- **Chi-Square Test of Independence**, because both variables are categorical.
#### **Contingency Table**

| Customer Segment | Not Smartphone | Smartphone |
|------------------|----------------|------------|
| Non-Premium      |      3800      |    874     |
| Premium          |      4142      |    933     |
#### **Chi-Square Test Results**
- **Chi² statistic** = 0.14  
- **p-value** = 0.709  
- **Degrees of freedom** = 1
#### **Conclusion**
Since the **p-value = 0.709 > 0.05**, we **fail to reject the null hypothesis**.  
This means there is **no statistically significant difference** in the number of smartphone purchases between the Premium and Non-Premium customer segments.

The data suggests that customer segment and smartphone purchases are **independent** of each other.


In [5]:
import pandas as pd
from scipy.stats import chi2_contingency

# Creating a new column that marks whether the gadget is a Smartphone or not
df_cleaned["Is_Smartphone"] = df_cleaned["Gadget Type"] == "Smartphone"

# Creating a contingency table (cross-tabulation)
contingency_table = pd.crosstab(df_cleaned["Customer Segment"], df_cleaned["Is_Smartphone"])

# Perform Chi-Square Test of Independence
chi2_stat, p_val, dof, expected = chi2_contingency(contingency_table)

# Printing the results
print("Contingency Table:")
print(contingency_table)

print("\nChi-square statistic:", chi2_stat)
print("p-value:", p_val)
print("Degrees of freedom:", dof)


Contingency Table:
Is_Smartphone     False  True 
Customer Segment              
Non-Premium        3800    874
Premium            4142    933

Chi-square statistic: 0.13966092001575528
p-value: 0.7086183365815202
Degrees of freedom: 1
